In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, col, explode, split, row_number
from pyspark.sql.window import Window
import re
import html
from prettytable import PrettyTable

spark = SparkSession.builder.appName('Lab2').master('local').getOrCreate()

In [11]:
languages = spark.read.csv('/content/programming-languages.csv', header=True)
languages = languages.select(lower(languages.name).alias('name'))

posts = spark.sparkContext.textFile('posts_sample.xml')

def extract_tags(line):
  return re.findall(r'<([^>]+)>', html.unescape(line))

def extract_year(line):
  return int(line.split('-')[0])

def extract_data(line):
  if not line.strip().startswith('<row'):
    return None

  year_match = re.search(r'CreationDate="([^"]+)"', line)
  if not year_match:
      return None
  year = extract_year(year_match.group(1))

  tags_match = re.search(r'Tags="([^"]+)"', line)
  if not tags_match:
    return None
  tags = extract_tags(tags_match.group(1))

  return (year, tags)

posts_data = posts \
  .map(extract_data) \
  .filter(lambda x: x is not None) \
  .toDF(['year', 'tags'])


processed_data = posts_data \
  .filter((col("year") >= "2010") & (col("year") <= "2020")) \
  .withColumn("tag", explode(col("tags"))) \
  .withColumn("tag", lower(col("tag"))) \
  .join(languages, col("tag") == col("name")) \
  .groupBy("year", "tag") \
  .count()

result = processed_data \
  .withColumn("rank", row_number().over(Window.partitionBy("year").orderBy(col("count").desc()))) \
  .filter(col("rank") <= 10) \
  .drop("rank")

result.write.mode("overwrite").parquet("result.parquet")
result_data = result.collect()
table = PrettyTable()
table.field_names = ["Год", "Язык", "Кол-во постов"]
for row in result_data:
  table.add_row([row['year'], row['tag'], row['count']])
print(table)

+------+-------------+---------------+
| Год  |     Язык    | Кол-во постов |
+------+-------------+---------------+
| 2010 |     java    |       52      |
| 2010 |     php     |       46      |
| 2010 |  javascript |       44      |
| 2010 |    python   |       26      |
| 2010 | objective-c |       23      |
| 2010 |      c      |       20      |
| 2010 |     ruby    |       12      |
| 2010 |    delphi   |       8       |
| 2010 |     bash    |       3       |
| 2010 |      r      |       3       |
| 2011 |     php     |      102      |
| 2011 |     java    |       93      |
| 2011 |  javascript |       83      |
| 2011 |    python   |       37      |
| 2011 | objective-c |       34      |
| 2011 |      c      |       24      |
| 2011 |     ruby    |       20      |
| 2011 |     perl    |       9       |
| 2011 |    delphi   |       8       |
| 2011 |     bash    |       7       |
| 2012 |     php     |      154      |
| 2012 |  javascript |      132      |
| 2012 |     java    |   